In [1]:
import pandas as pd

from pathlib import Path

from kpi import calculate_all_kpis

In [2]:
DATA_PATH = Path("../data/preprocess/clean_call_center.csv")

df = pd.read_csv(DATA_PATH)

df["call_date"] = pd.to_datetime(df["call_date"])

df.head()

,call_id,call_date,start_time,customer_phone,call_reason,agent_id,queue,channel,status,wait_seconds,...,hold_seconds,after_call_work_seconds,csat_score,first_contact_resolution,transfer_count,call_datetime,hour,week_day,day,shift_time
0,C0013,2026-01-01,0 days 08:32:04,98912000790,Account Access,A01,Support,Phone,Answered,106,...,16.0,59.0,3.0,Yes,0,2026-01-01 08:32:04,8,Thursday,1,Morning
1,C0029,2026-01-01,0 days 08:55:30,912000134,Account Access,A06,Sales,Phone,Answered,11,...,13.0,42.0,2.0,Yes,0,2026-01-01 08:55:30,8,Thursday,1,Morning
2,C0032,2026-01-01,0 days 10:22:55,912000374,Subscription Change,A06,Billing,Callback,Answered,67,...,56.0,67.0,3.0,No,0,2026-01-01 10:22:55,10,Thursday,1,Morning
3,C0033,2026-01-01,0 days 10:44:31,912000295,Product Inquiry,A05,Support,Phone,Abandoned,163,...,NaN,NaN,NaN,NaN,0,2026-01-01 10:44:31,10,Thursday,1,Morning
4,C0028,2026-01-01,0 days 11:00:39,912000641,Account Access,A03,Support,Phone,Answered,106,...,13.0,53.0,3.0,No,1,2026-01-01 11:00:39,11,Thursday,1,Morning


# Anlaysis Data

In [3]:
overall_kpis = calculate_all_kpis(df)

overall_kpis

{'answer_rate': np.float64(87.29166666666667),
 'abandon_rate': np.float64(12.708333333333332),
 'awt': np.float64(95.0),
 'aht': np.float64(398.07398568019096),
 'csat_avg': np.float64(3.5823389021479715),
 'csat_pct': np.float64(56.56324582338902),
 'fcr': 43.198090692124104}

In [4]:
print(df.shape)
df["status"].value_counts()

(960, 21)


status
Answered       733
Abandoned      122
Transferred    105
Name: count, dtype: int64

In [5]:
handled_calls = ["Answered", "Transferred"]

handled_df = df[df["status"].isin(handled_calls)].copy()

In [7]:
team_kpis = pd.DataFrame({team: calculate_all_kpis(group)
                          for team, group in df.groupby("queue")}).T

team_kpis

,answer_rate,abandon_rate,awt,aht,csat_avg,csat_pct,fcr
Billing,89.565217,10.434783,68.0,318.286408,3.825243,64.563107,48.058252
Sales,92.682927,7.317073,53.0,324.436090,3.973684,74.436090,44.736842
Support,82.618510,17.381490,141.0,496.500000,3.161202,39.071038,39.344262


In [8]:
shift_kpis = pd.DataFrame({shift: calculate_all_kpis(group)
                           for shift, group in df.groupby("shift_time")}).T

shift_kpis

,answer_rate,abandon_rate,awt,aht,csat_avg,csat_pct,fcr
Afternoon,86.407767,13.592233,95.0,397.554307,3.601124,56.741573,43.820225
Morning,88.888889,11.111111,95.0,398.986842,3.549342,56.250000,42.105263


In [9]:
reason_counts = df["call_reason"].value_counts().rename_axis("call_reason").reset_index(name="calls")

reason_counts

,call_reason,calls
0,Technical Problem,206
1,Account Access,157
2,Product Inquiry,122
3,Order Status,114
4,Billing Issue,107
5,Complaint,99
6,Subscription Change,79
7,Refund Request,76


In [10]:
reason_kpis = pd.DataFrame({reason: calculate_all_kpis(group)
                            for reason, group in df.groupby("call_reason")}).T

reason_kpis

,answer_rate,abandon_rate,awt,aht,csat_avg,csat_pct,fcr
Account Access,85.350318,14.649682,98.0,405.619403,3.664179,57.462687,41.044776
Billing Issue,92.523364,7.476636,85.0,371.474747,3.595960,53.535354,46.464646
Complaint,88.888889,11.111111,96.0,411.931818,3.329545,52.272727,44.318182
Order Status,87.719298,12.280702,82.0,390.040000,3.730000,63.000000,49.000000
Product Inquiry,87.704918,12.295082,81.0,371.962617,3.570093,55.140187,39.252336
Refund Request,82.894737,17.105263,84.0,369.158730,3.841270,73.015873,58.730159
Subscription Change,94.936709,5.063291,90.0,371.413333,3.666667,58.666667,37.333333
Technical Problem,83.495146,16.504854,121.0,443.546512,3.430233,50.000000,38.372093


In [11]:
team_reason = (
    df.groupby(["queue", "call_reason"])
      .agg(
          calls=("call_id", "count"),
          avg_wait=("wait_seconds", "mean"),
          avg_handle=("handle_seconds", "mean"),
          avg_csat=("csat_score", "mean")
      )
      .reset_index()
)

team_reason.sort_values("avg_wait", ascending=False).head(10)

,queue,call_reason,calls,avg_wait,avg_handle,avg_csat
0,Billing,Account Access,40,68.525000,225.805556,4.027778
1,Billing,Billing Issue,61,65.868852,230.642857,3.892857
2,Billing,Complaint,24,70.458333,241.857143,3.142857
3,Billing,Order Status,16,61.000000,240.692308,3.769231
4,Billing,Product Inquiry,10,73.700000,216.000000,3.333333
5,Billing,Refund Request,35,62.285714,210.187500,4.156250
6,Billing,Subscription Change,18,85.000000,255.933333,3.600000
7,Billing,Technical Problem,26,69.346154,217.375000,3.875000
8,Sales,Account Access,34,49.529412,234.322581,4.096774
9,Sales,Billing Issue,15,54.466667,214.000000,3.714286


In [12]:
shift_reason = (
    df.groupby(["shift_time", "call_reason"])
      .agg(
          calls=("call_id", "count"),
          avg_wait=("wait_seconds", "mean"),
          avg_handle=("handle_seconds", "mean"),
          avg_csat=("csat_score", "mean")
      )
      .reset_index()
)

shift_reason.sort_values("avg_wait", ascending=False).head(10)

,shift_time,call_reason,calls,avg_wait,avg_handle,avg_csat
7,Afternoon,Technical Problem,136,129.602941,352.241071,3.446429
15,Morning,Technical Problem,70,114.742857,360.533333,3.400000
8,Morning,Account Access,54,103.611111,310.921569,3.470588
2,Afternoon,Complaint,69,101.275362,321.838710,3.360656
0,Afternoon,Account Access,103,100.766990,310.511905,3.783133
6,Afternoon,Subscription Change,54,93.500000,274.220000,3.700000
10,Morning,Complaint,30,91.866667,313.851852,3.259259
11,Morning,Order Status,39,91.769231,274.967742,3.580645
13,Morning,Refund Request,30,89.600000,269.791667,3.583333
12,Morning,Product Inquiry,48,89.416667,300.023810,3.738095


---

# گزارش مدیریتی

<div dir="rtl" style="text-align: right;">
    
## تحلیل عملکرد مرکز تماس

* **تیم Support ضعیف‌ترین عملکرد کلی را در میان تیم‌ها دارد.** این تیم بالاترین نرخ رها کردن تماس (17.38%)، بالاترین زمان انتظار متوسط (141 ثانیه) و بالاترین زمان رسیدگی متوسط (496.5 ثانیه) را دارد. همچنین CSAT متوسط آن 3.16 و FCR آن 39.34% است که پایین‌تر از دو تیم دیگر قرار دارد.

* **نوع تماس یکی از عوامل محتمل در عملکرد ضعیف‌تر Support است.** در Support، زمان انتظار و زمان رسیدگی برای تقریباً تمام انواع تماس بالا است؛ برای مثال AHT تماس‌های `Order Status`، `Billing Issue` و `Technical Problem` به‌ترتیب حدود 416، 405 و 413 ثانیه است. این نشان می‌دهد مسئله صرفاً به یک نوع تماس محدود نیست و احتمالاً پیچیدگی یا فرآیند رسیدگی در این تیم نقش دارد.

* **Technical Problem مهم‌ترین حوزه برای بررسی و بهبود است.** این نوع تماس با 206 مورد، پرتکرارترین دلیل تماس در کل داده است و 149 مورد از آن مربوط به Support است. همچنین Technical Problem دارای AWT برابر 121 ثانیه و AHT برابر 443.5 ثانیه است که از بسیاری از انواع تماس بالاتر است. بنابراین تمرکز بر بهبود فرآیند رسیدگی به این نوع تماس می‌تواند اثر قابل‌توجهی بر عملکرد کلی داشته باشد.

* **تفاوت عملکرد بین شیفت‌های Morning و Afternoon محدود است.** AHT در دو شیفت تقریباً برابر است (حدود 399 و 398 ثانیه) و اختلاف Answer Rate و Abandon Rate نیز نسبتاً کم است. بنابراین بر اساس این داده‌ها، تمرکز اصلی اقدامات اصلاحی بهتر است روی تیم و نوع تماس باشد، نه صرفاً تغییر شیفت کاری.




<div dir="rtl" style="text-align: right;">

## پیشنهادهای بهبود

* برای تماس‌های **Technical Problem**، ایجاد راهنمای عیب‌یابی استاندارد، پایگاه دانش و اسکریپت‌های پاسخ می‌تواند زمان رسیدگی و وابستگی به حل دستی مشکلات را کاهش دهد.
* عملکرد Support در **AWT، AHT و FCR** به‌صورت جداگانه پایش شود و برای تماس‌های پرتکرار این تیم، آموزش تخصصی و فرآیندهای استاندارد حل مسئله تعریف شود.
* برای کاهش Abandon Rate در Support، ظرفیت و تخصیص Agentها بر اساس حجم و نوع تماس بررسی شود، به‌خصوص در بازه‌هایی که Technical Problem حجم بالاتری دارد.
* در ادامه، اثر این اقدامات با مقایسه دوره‌های زمانی بعدی در KPIهای **AHT، Abandon Rate، CSAT و FCR** ارزیابی شود.